# Stage 30A: smooth residual TCN

Stage 29Aと同じ1,370 cuts・固定weight 0.20で、平滑base残差をdilated TCNとして学習します。L4/A100推奨、T4も使用可能です。予約120 wellsは未使用です。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess,sys
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
import torch
assert torch.cuda.is_available(),'Select a GPU runtime before running Stage 30A'
print(torch.cuda.get_device_name(0))


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage21b_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
manifest_run=artifact_dir/'stage29a_multicut_manifest_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',public_oof_run/'base_oof.parquet',stage21b_run/'confidence_cut_report.parquet',manifest_run/'training_cut_ids.parquet',manifest_run/'confirmation_cut_ids.parquet']
for path in required: assert path.is_file(),path
manifest=json.loads((manifest_run/'summary.json').read_text())
assert manifest['training_wells']==500 and manifest['confirmation_wells']==120 and manifest['reserved_confirmation_used'] is False,manifest
print(manifest['training_cuts'],'training cuts; confirmation reserve remains sealed')


In [ ]:
RUN_ID='stage30a_residual_tcn_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve(); assert resolved==expected and resolved.parent==artifact_dir.resolve(); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    command=[sys.executable,'-m','rogii.cli.sequence_residual_field','--config','configs/experiment/stage30a_residual_tcn.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--split-manifest-run',str(manifest_run),'--validation-run',str(stage21b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID,'--device','cuda']
    training_env=os.environ.copy(); training_env['PYTHONPATH']=str(repo_dir/'src')+':'+training_env.get('PYTHONPATH','')
    log_path=artifact_dir/(RUN_ID+'_driver.log')
    with log_path.open('w') as log:
        process=subprocess.Popen(command,cwd=repo_dir,env=training_env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
        for line in process.stdout: print(line,end=''); log.write(line); log.flush()
        return_code=process.wait()
    if return_code: raise RuntimeError(f'Stage 30A failed with exit code {return_code}. Log: {log_path}')
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage30a_complete','promoted_to_stage30b_reserved_confirmation','device','training_cuts','training_wells','training_rows','validation_cuts','validation_wells','feature_count','ensemble_models','primary_weight','base_rmse','candidate_rmse','rmse_delta','bootstrap_95pct','cut_rmse_p90_delta','gates','reserved_confirmation_used','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['weight_report']).sort_values('weight')
